In [1]:
import torch
from sklearn import linear_model
import os
import numpy as np
from utils import ProcessFoldData
from scipy.stats import wilcoxon
from utils import SVD

import warnings 
warnings.filterwarnings("ignore")

# DEFAULT FILE PATHS
datasets = "../datasets/"
output = "../output/"
# datasets = "../datasets/layers10_big"
try: os.mkdir(output)
except: pass

print("Default file paths:-------------")
print(f"datasets: {datasets}")
print(f"output: {output}")
print("--------------------------------")



# DEFAULT PARAMETERS
nLambdas = 10
minLambda = 0.0001
maxLambda = 3.5
K = 4            # no of folds used for cross validation
sigThresh = .05   # sigma threshold
maxiter = 200

print("Default parameters:-------------")
print(f"nLambdas: {nLambdas}")
print(f"minLambda: {minLambda}")
print(f"maxLambda: {maxLambda}")
print(f"Number of folds: {K}")
print(f"sigThresh: {sigThresh}")
print(f"maxiter: {maxiter}")
print("--------------------------------")



# PYTORCH ENVIRONMENT VARIABLES
# device = torch.device("cuda")
device = torch.device("cpu")

print(f"Device: {device}")



# DATASET PARAMETERS FOR RANDOM DATA
Bsz = 100
Edim = 384
Fdim = 21

# Loading from a file
file = True
if file:
  Embeddings = torch.tensor(np.genfromtxt(f"{datasets}/embeddings.csv", delimiter=',', dtype='float64'), device=device)
  Features =  torch.tensor(np.genfromtxt(f"{datasets}/features.csv", delimiter=',', skip_header=1, dtype='float64'), device=device)

else:
  Embeddings = torch.rand(Bsz,Edim).to(torch.float64).to(device)
  Features = torch.rand(Bsz,Fdim).to(torch.float64).to(device)


assert Embeddings.shape[1] == Edim
# assert Features.shape[1] == Fdim


##############################################
#### Run BIOT for different lambda values ####
##############################################

print("Selection of lambda in progress...")

# Define lambda vector, feature vector, and embedding vector
lambdaVals = torch.exp(torch.linspace(np.log(minLambda), np.log(maxLambda), nLambdas)).to(device) / np.sqrt(Features.shape[1])
print(f"Lambda values: {lambdaVals}")

# Split data into K folds such that each foldid has the indexes to use
foldIds = torch.split(torch.randperm(Features.size(0)), Features.size(0) // K)


Default file paths:-------------
datasets: ../datasets/
output: ../output/
--------------------------------
Default parameters:-------------
nLambdas: 10
minLambda: 0.0001
maxLambda: 3.5
Number of folds: 4
sigThresh: 0.05
maxiter: 200
--------------------------------
Device: cpu
Selection of lambda in progress...
Lambda values: tensor([2.1822e-05, 6.9789e-05, 2.2319e-04, 7.1380e-04, 2.2828e-03, 7.3008e-03,
        2.3349e-02, 7.4673e-02, 2.3882e-01, 7.6376e-01])


In [2]:
Embeddings.shape

torch.Size([30, 384])

In [3]:
Features.shape

torch.Size([30, 21])

In [16]:
torch.sum(Features**2, dim=0)

tensor([357.0000,  46.7371,  27.1715,  23.7653,  23.2921,  25.4810,  15.6329,
         21.5407,  14.9046,  24.8161,  24.5085,  17.8504,  14.3371,  54.7662,
         28.6984,  18.1304,  33.0780,  39.0125,  12.8239,  26.1161,  26.3441],
       dtype=torch.float64)

In [2]:
Embeddings.shape, Features.shape, (Edim, Fdim), 

(torch.Size([30, 384]), torch.Size([30, 21]), (384, 21))

In [3]:
clf = linear_model.Lasso(alpha=0.1, fit_intercept=False)
# intializing the rotation matrix
clf.coef_ = torch.zeros(Edim, 19, dtype=torch.float64, device=device)

In [4]:
# preprocess embeddings and features
Features_norm, Embeddings_norm, Features_test, Embeddings_test = ProcessFoldData(X = Embeddings, Fe = Features, testId = foldIds[1], CV=True)

print(F"Training data size {Features_norm.size()} and Testing data size {Features_test.size()}")


Training data size torch.Size([23, 21]) and Testing data size torch.Size([7, 21])


In [5]:
clf.coef_.T.shape

torch.Size([19, 384])

In [6]:

for iter in range(1000):
    W = torch.tensor( clf.coef_.T, device=Embeddings.device)
    Rotation = SVD(Features_norm, W, Embeddings_norm)
    
    Y = torch.matmul(Embeddings_norm,Rotation)
    print(Y.shape)
    break
    clf.fit(Features_norm.cpu(),Y.cpu())
    
    
    Yp= clf.predict(Features_test.cpu())
    Y = torch.matmul(Embeddings_test,Rotation)
    print(clf.score(Features_test.cpu(), Y.cpu()))

    mse = torch.mean((Y.cpu() - Yp)**2) 
    reg = np.sum(np.abs(clf.coef_))
    mse_error = mse + reg
    print(f"Iteration {iter} : MSE error {mse_error} MSE {mse} Reg {reg}")
    print(W.sum(), Rotation.sum())

RuntimeError: mat1 and mat2 shapes cannot be multiplied (23x21 and 19x384)

In [1]:
import torch
import numpy as np

def coordinate_descent(X, y, beta=None, max_iter=1000, tol=1e-4):
    _, p = X.shape
    if beta == None: 
        beta = torch.zeros(p)
    
    for _ in range(max_iter):
        beta_old = beta.clone()
        
        for j in range(p):
            # Calculate the residual
            r = y - torch.mv(X, beta)
            
            # Add back the current feature's contribution
            r += X[:, j] * beta[j]
            
            # Update the coefficient
            beta[j] = torch.dot(X[:, j], r) / torch.dot(X[:, j], X[:, j])
        
        # Check for convergence
        if torch.norm(beta - beta_old) < tol:
            break
    
    return beta

# Generate sample data
np.random.seed(42)
n_samples, n_features = 100, 5
X = np.random.randn(n_samples, n_features)
true_beta = np.array([1, 2, 3, 4, 5])
y = X.dot(true_beta) + np.random.randn(n_samples) * 0.1

# Convert to PyTorch tensors
X_torch = torch.tensor(X, dtype=torch.float32)
y_torch = torch.tensor(y, dtype=torch.float32)

# Run coordinate descent
beta_hat = coordinate_descent(X_torch, y_torch)

print("Estimated coefficients:", beta_hat)
print("True coefficients:", true_beta)

Estimated coefficients: tensor([1.0058, 2.0123, 2.9994, 4.0142, 4.9998])
True coefficients: [1 2 3 4 5]


In [35]:
import torch
import numpy as np

def lasso_coordinate_descent(X, y, alpha, max_iter=1, tol=1e-4):
    _, p = X.shape # 100, 10
    beta = torch.ones(p) # 10
  
    for _ in range(max_iter):
        beta_old = beta.clone()
        
        for j in range(p):
            # Calculate the residual
            r = y - torch.mv(X, beta) # 100 
            # Add back the current feature's contribution
            r += X[:, j] * beta[j] # 100
            
            # Calculate the update
            z_j = torch.dot(X[:, j], r)
            theta_j = torch.dot(X[:, j], X[:, j])
            
            if z_j < -alpha / 2:
                beta[j] = (z_j + alpha / 2) / theta_j
            elif z_j > alpha / 2:
                beta[j] = (z_j - alpha / 2) / theta_j
            else:
                beta[j] = 0
        
        # Check for convergence
        if torch.norm(beta - beta_old) < tol:
            break
    
    return beta

# Generate sample data
np.random.seed(42)
n_samples, n_features = 2, 10
X = np.random.randn(n_samples, n_features)
true_beta = np.array([1, 2, 0, 0, 3, 0, 0, 4, 0, 5])
y = X.dot(true_beta) + np.random.randn(n_samples) * 0.1

# Convert to PyTorch tensors
X_torch = torch.tensor(X, dtype=torch.float32)
y_torch = torch.tensor(y, dtype=torch.float32)

# Run Lasso regression with coordinate descent
alpha = 4  # Lasso penalty parameter
beta_hat = lasso_coordinate_descent(X_torch, y_torch, alpha)

print("Estimated coefficients:", beta_hat)
print("True coefficients:", true_beta)

Estimated coefficients: tensor([2.2151, 0.0628, 0.0000, 2.2085, 1.4441, 0.0000, 0.4048, 0.0000, 0.4546,
        1.6862])
True coefficients: [1 2 0 0 3 0 0 4 0 5]


: 

In [ ]:
torch.mv(X, beta)

In [5]:
import torch
import numpy as np

def lasso_coordinate_descent(X, Y, alpha, beta=None, max_iter=1000, tol=1e-4):
    _, p = X.shape
    _, q = Y.shape
    
    if beta == None: 
        beta = torch.zeros((p, q)).to(X.device)
    
    for _ in range(max_iter):
        beta_old = beta.clone()
        
        for j in range(p):
            for k in range(q):
                # Calculate the residual
                r = Y[:, k] - torch.mv(X, beta[:, k])
                
                # Add back the current feature's contribution
                r += X[:, j] * beta[j, k]
                
                # Calculate the update
                z_j = torch.dot(X[:, j], r)
                theta_j = torch.dot(X[:, j], X[:, j])
                
                if z_j < -alpha / 2:
                    beta[j, k] = (z_j + alpha / 2) / theta_j
                elif z_j > alpha / 2:
                    beta[j, k] = (z_j - alpha / 2) / theta_j
                else:
                    beta[j, k] = 0
        
        # Check for convergence
        if torch.norm(beta - beta_old) < tol:
            break
    
    return beta

# Generate sample data
np.random.seed(42)
n_samples, n_features, n_targets = 100, 10, 3
X = np.random.randn(n_samples, n_features)
true_beta = np.array([[1, 0.5, 2],
                      [2, 0, 1],
                      [0, 1.5, 0],
                      [0, 0, 0],
                      [3, 2, 1],
                      [0, 0, 0],
                      [0, 1, 0],
                      [4, 0, 3],
                      [0, 0.5, 0],
                      [5, 3, 2]])
Y = X.dot(true_beta) + np.random.randn(n_samples, n_targets) * 0.1

# Convert to PyTorch tensors
X_torch = torch.tensor(X, dtype=torch.float32)
Y_torch = torch.tensor(Y, dtype=torch.float32)

# Run Lasso regression with coordinate descent
alpha = 10  # Lasso penalty parameter
beta_hat = lasso_coordinate_descent(X_torch, Y_torch, alpha, beta=None)

print("Estimated coefficients:")
print(beta_hat)
print("\nTrue coefficients:")
print(true_beta)

Estimated coefficients:
tensor([[0.9215, 0.4208, 1.9320],
        [1.9348, 0.0000, 0.9289],
        [0.0000, 1.4557, 0.0000],
        [0.0000, 0.0000, 0.0000],
        [2.9696, 1.9829, 0.9556],
        [0.0000, 0.0000, 0.0000],
        [0.0000, 0.9341, 0.0000],
        [3.9423, 0.0000, 2.9620],
        [0.0000, 0.4170, 0.0000],
        [4.9506, 2.9407, 1.9353]])

True coefficients:
[[1.  0.5 2. ]
 [2.  0.  1. ]
 [0.  1.5 0. ]
 [0.  0.  0. ]
 [3.  2.  1. ]
 [0.  0.  0. ]
 [0.  1.  0. ]
 [4.  0.  3. ]
 [0.  0.5 0. ]
 [5.  3.  2. ]]


In [14]:
import torch

def lasso_coordinate_descent(X, Y, alpha, beta=None, max_iter=1000, tol=1e-4):
    _, p = X.shape
    _, q = Y.shape
    
    if beta == None: 
        beta = torch.zeros((p, q)).to(X.device)
    
    for _ in range(max_iter):
        beta_old = beta.clone()
        
        for j in range(p):
            for k in range(q):
                # Calculate the residual
                r = Y[:, k] - torch.mv(X, beta[:, k])
                
                # Add back the current feature's contribution
                r += X[:, j] * beta[j, k]
                
                # Calculate the update
                z_j = torch.dot(X[:, j], r)
                theta_j = torch.dot(X[:, j], X[:, j])
                
                if z_j < -alpha / 2:
                    beta[j, k] = (z_j + alpha / 2) / theta_j
                elif z_j > alpha / 2:
                    beta[j, k] = (z_j - alpha / 2) / theta_j
                else:
                    beta[j, k] = 0
        
        # Check for convergence
        if torch.norm(beta - beta_old) < tol:
            break
    
    return beta


alpha = 1
for iter in range(1000):
    W = torch.zeros(Edim, Fdim).to(torch.float32).T
    Rotation = SVD(Features_norm, W, Embeddings_norm)
    
    Y = torch.matmul(Embeddings_norm,Rotation)
    W = lasso_coordinate_descent(Features_norm.cpu(),Y.cpu(), alpha, beta=W)

    Yp= Features_test @ W
    Y = torch.matmul(Embeddings_test,Rotation)


    mse = torch.mean((Y.cpu() - Yp)**2) 
    reg = torch.sum(torch.abs(W))
    mse_error = mse + reg
    print(f"Iteration {iter} : MSE error {mse_error} MSE {mse} Reg {reg}")
    print(W.sum(), Rotation.sum())

Iteration 0 : MSE error 0.693247377872467 MSE 0.0010282945586368442 Reg 0.692219078540802
tensor(-0.0792) tensor(384.)
Iteration 1 : MSE error 0.693247377872467 MSE 0.0010282945586368442 Reg 0.692219078540802
tensor(-0.0792) tensor(384.)
Iteration 2 : MSE error 0.693247377872467 MSE 0.0010282945586368442 Reg 0.692219078540802
tensor(-0.0792) tensor(384.)


KeyboardInterrupt: 

tensor(0.0012)

[2, 3]

: 

In [92]:
import torch
a = torch.rand(4)*5
c = torch.rand(4)*5
b = 3
a,c

(tensor([1.0231, 2.3051, 4.9480, 0.1393]),
 tensor([4.4577, 2.2230, 4.3538, 2.8381]))

In [93]:
c < -2

tensor([False, False, False, False])

In [94]:
a[2,:][c < -2]

IndexError: too many indices for tensor of dimension 1

In [95]:
a[2,:][c < -2] = 0
a[2,:]

IndexError: too many indices for tensor of dimension 1

In [96]:
c < b

tensor([False,  True, False,  True])

In [97]:
a

tensor([1.0231, 2.3051, 4.9480, 0.1393])

In [113]:
w = torch.abs(a-c).unsqueeze(-1)
w.shape

torch.Size([4, 1])

In [115]:
w

tensor([[3.4346],
        [0.0821],
        [0.5942],
        [2.6988]])

In [116]:
torch.max(w, torch.ones_like(w))

tensor([[3.4346],
        [1.0000],
        [1.0000],
        [2.6988]])

In [86]:
a = torch.rand(4,5)
a

tensor([[0.9406, 0.4864, 0.9297, 0.0249, 0.7363],
        [0.1388, 0.9347, 0.1176, 0.0206, 0.3176],
        [0.9159, 0.5491, 0.0542, 0.6891, 0.7389],
        [0.3025, 0.2498, 0.4666, 0.7141, 0.5801]])

In [89]:
torch.sum(a**2, dim=0).shape

torch.Size([5])

In [ ]:
import numpy as np
import warnings
from sklearn.exceptions import ConvergenceWarning

def enet_coordinate_descent(w, alpha, beta, X, y, max_iter, tol, rng, random=False, positive=False):
    """NumPy version of the coordinate descent algorithm for Elastic-Net regression

    We minimize:
    (1/2) * norm(y - X w, 2)^2 + alpha norm(w, 1) + (beta/2) norm(w, 2)^2

    Parameters:
    -----------
    w : ndarray, shape (n_features,)
        Coefficient vector
    alpha : float
        Constant that multiplies the L1 term
    beta : float
        Constant that multiplies the L2 term
    X : ndarray, shape (n_samples, n_features)
        Training data
    y : ndarray, shape (n_samples,)
        Target values
    max_iter : int
        Maximum number of iterations
    tol : float
        Tolerance for the optimization
    rng : numpy.random.Generator
        Random number generator
    random : bool, default=False
        Whether to use random coordinate descent
    positive : bool, default=False
        If set to True, forces coefficients to be positive

    Returns:
    --------
    w : ndarray, shape (n_features,)
        Elastic-Net coefficients
    gap : float
        Achieved dual gap
    tol : float
        Tolerance used for the dual gap
    n_iter : int
        Number of coordinate descent iterations
    """

    dtype = X.dtype
    n_samples, n_features = X.shape

    if alpha == 0 and beta == 0:
        warnings.warn("Coordinate descent with no regularization may lead to unexpected results and is discouraged.")

    # Compute norms of the columns of X
    norm_cols_X = np.sum(X**2, axis=0)

    # Initial value of the residuals
    R = y - X.dot(w)

    # Adjust tolerance
    tol *= np.dot(y, y)

    for n_iter in range(max_iter):
        w_max = 0.0
        d_w_max = 0.0
        
        for f_iter in range(n_features):
            if random:
                ii = rng.integers(n_features)
            else:
                ii = f_iter

            if norm_cols_X[ii] == 0.0:
                continue

            w_ii = w[ii]  # Store previous value

            if w_ii != 0.0:
                R += w_ii * X[:, ii]

            tmp = X[:, ii].dot(R)

            if positive and tmp < 0:
                w[ii] = 0.0
            else:
                w[ii] = np.sign(tmp) * max(abs(tmp) - alpha, 0) / (norm_cols_X[ii] + beta)

            if w[ii] != 0.0:
                R -= w[ii] * X[:, ii]  # Update residual

            # Update the maximum absolute coefficient update
            d_w_ii = abs(w[ii] - w_ii)
            d_w_max = max(d_w_max, d_w_ii)

            w_max = max(w_max, abs(w[ii]))

        if w_max == 0.0 or d_w_max / w_max < tol or n_iter == max_iter - 1:
            # Check the duality gap as ultimate stopping criterion

            XtA = X.T.dot(R) - beta * w

            if positive:
                dual_norm_XtA = np.max(XtA)
            else:
                dual_norm_XtA = np.max(np.abs(XtA))

            R_norm2 = R.dot(R)
            w_norm2 = w.dot(w)

            if dual_norm_XtA > alpha:
                const = alpha / dual_norm_XtA
                A_norm2 = R_norm2 * (const ** 2)
                gap = 0.5 * (R_norm2 + A_norm2)
            else:
                const = 1.0
                gap = R_norm2

            l1_norm = np.sum(np.abs(w))

            gap += (alpha * l1_norm - const * np.dot(R, y) + 0.5 * beta * (1 + const ** 2) * w_norm2)

            if gap < tol:
                break
    else:
        message = (
            "Objective did not converge. You might want to increase "
            "the number of iterations, check the scale of the "
            "features or consider increasing regularisation. "
            f"Duality gap: {gap:.3e}, tolerance: {tol:.3e}"
        )
        if alpha < np.finfo(np.float64).eps:
            message += (
                " Linear regression models with null weight for the "
                "l1 regularization term are more efficiently fitted "
                "using one of the solvers implemented in "
                "sklearn.linear_model.Ridge/RidgeCV instead."
            )
        warnings.warn(message, ConvergenceWarning)

    return w, gap, tol, n_iter + 1

In [1]:
import torch
import warnings
from sklearn.exceptions import ConvergenceWarning

def enet_coordinate_descent_multi_task(
    W,
    l1_reg,
    l2_reg,
    X,
    Y,
    max_iter,
    tol,
    random=False
):
    """PyTorch version of the coordinate descent algorithm
    for Elastic-Net multi-task regression

    We minimize:
    0.5 * norm(Y - X W.T, 2)^2 + l1_reg ||W.T||_21 + 0.5 * l2_reg norm(W.T, 2)^2

    Parameters:
    -----------
    W : torch.Tensor of shape (n_tasks, n_features)
        Initial coefficients
    l1_reg : float
        L1 regularization parameter
    l2_reg : float
        L2 regularization parameter
    X : torch.Tensor of shape (n_samples, n_features)
        Input data
    Y : torch.Tensor of shape (n_samples, n_tasks)
        Target values
    max_iter : int
        Maximum number of iterations
    tol : float
        Tolerance for the optimization
    random : bool, default=False
        Whether to use random coordinate descent

    Returns:
    --------
    W : torch.Tensor of shape (n_tasks, n_features)
        Elastic-Net coefficients
    gap : float
        Achieved dual gap
    tol : float
        Tolerance used for the dual gap
    n_iter : int
        Number of coordinate descent iterations
    """

    device = X.device
    dtype = X.dtype
    n_samples, n_features = X.shape
    n_tasks = Y.shape[1]

    if l1_reg == 0:
        warnings.warn(
            "Coordinate descent with l1_reg=0 may lead to unexpected"
            " results and is discouraged."
        )

    # Compute norms of the columns of X
    norm_cols_X = torch.sum(X**2, dim=0)

    # Initial residuals
    R = Y - X @ W.T

    # Adjust tolerance
    tol = tol * torch.norm(Y, p='fro')**2

    for n_iter in range(max_iter):
        w_max = 0.0
        d_w_max = 0.0

        for f_iter in range(n_features):
            if random:
                ii = torch.randint(n_features, (1,)).item()
            else:
                ii = f_iter

            if norm_cols_X[ii] == 0.0:
                continue

            w_ii = W[:, ii].clone()  # Store previous value
            
            if torch.any(w_ii != 0):
                R += X[:, ii].unsqueeze(1) @ w_ii.unsqueeze(0)

            tmp = X[:, ii] @ R

            nn = torch.norm(tmp)

            if nn > l1_reg:
                W[:, ii] = tmp * (1 - l1_reg / nn) / (norm_cols_X[ii] + l2_reg)
            else:
                W[:, ii] = 0.0

            if torch.any(W[:, ii] != 0):
                R -= X[:, ii].unsqueeze(1) @ W[:, ii].unsqueeze(0)

            # Update the maximum absolute coefficient update
            d_w_ii = torch.max(torch.abs(W[:, ii] - w_ii))
            d_w_max = max(d_w_max, d_w_ii.item())

            w_max = max(w_max, torch.max(torch.abs(W[:, ii])).item())

        if w_max == 0.0 or d_w_max / w_max < tol or n_iter == max_iter - 1:
            # Check the duality gap as ultimate stopping criterion

            XtA = X.T @ R - l2_reg * W.T
            dual_norm_XtA = torch.max(torch.norm(XtA, dim=1))

            R_norm = torch.norm(R, p='fro')
            w_norm = torch.norm(W, p='fro')

            if dual_norm_XtA > l1_reg:
                const = l1_reg / dual_norm_XtA
                A_norm = R_norm * const
                gap = 0.5 * (R_norm**2 + A_norm**2)
            else:
                const = 1.0
                gap = R_norm**2

            ry_sum = torch.sum(R * Y)
            l21_norm = torch.sum(torch.norm(W, dim=0))

            gap += (
                l1_reg * l21_norm
                - const * ry_sum
                + 0.5 * l2_reg * (1 + const**2) * (w_norm**2)
            )

            if gap < tol:
                break
    else:
        warnings.warn("Objective did not converge. You might want to "
                      "increase the number of iterations. Duality "
                      f"gap: {gap}, tolerance: {tol}",
                      ConvergenceWarning)

    return W, gap, tol, n_iter + 1

In [ ]:
import torch
from sklearn import linear_model
import os
import numpy as np
from utils import ProcessFoldData
from scipy.stats import wilcoxon
from utils import SVD
from utils import ProcessFoldData, MSE, scale, SVD, L1
from tqdm.auto import tqdm

import warnings 
warnings.filterwarnings("ignore")

# DEFAULT FILE PATHS
datasets = "../datasets/"
output = "../output/"
datasets = "../datasets/layers10_big"
try: os.mkdir(output)
except: pass

print("Default file paths:-------------")
print(f"datasets: {datasets}")
print(f"output: {output}")
print("--------------------------------")



# DEFAULT PARAMETERS
nLambdas = 10
minLambda = 0.0001
maxLambda = 3.5
K = 10            # no of folds used for cross validation
sigThresh = .05   # sigma threshold
maxiter = 200

print("Default parameters:-------------")
print(f"nLambdas: {nLambdas}")
print(f"minLambda: {minLambda}")
print(f"maxLambda: {maxLambda}")
print(f"Number of folds: {K}")
print(f"sigThresh: {sigThresh}")
print(f"maxiter: {maxiter}")
print("--------------------------------")



# PYTORCH ENVIRONMENT VARIABLES
# device = torch.device("cuda")
device = torch.device("cuda")

print(f"Device: {device}")



# DATASET PARAMETERS FOR RANDOM DATA
Bsz = 20000
Edim = 384
Fdim = 21

# Loading from a file
file = True
# file = False
if file:
  Embeddings = torch.tensor(np.genfromtxt(f"{datasets}/embeddings.csv", delimiter=',', dtype='float64'), device=device)
  Features =  torch.tensor(np.genfromtxt(f"{datasets}/features.csv", delimiter=',', skip_header=1, dtype='float64'), device=device)
else:
  Embeddings = torch.rand(Bsz,Edim).to(torch.float64).to(device)
  Features = torch.rand(Bsz,Fdim).to(torch.float64).to(device)


Edim = Embeddings.shape[1]
Fdim = Features.shape[1] 


##############################################
#### Run BIOT for different lambda values ####
##############################################

print("Selection of lambda in progress...")

# Define lambda vector, feature vector, and embedding vector
lambdaVals = torch.exp(torch.linspace(np.log(minLambda), np.log(maxLambda), nLambdas)).to(device) / np.sqrt(Features.shape[1])
print(f"Lambda values: {lambdaVals}")

# Split data into K folds such that each foldid has the indexes to use
foldIds = torch.split(torch.randperm(Features.size(0)), Features.size(0) // K)


# preprocess embeddings and features
Features_norm, Embeddings_norm = ProcessFoldData(X = Embeddings, Fe = Features, testId = foldIds[0], only_standardize=True)
# Features_norm, Embeddings_norm, Features_test, Embeddings_test = ProcessFoldData(X = Embeddings, Fe = Features, testId = foldIds[1], CV=True)

print(F"Training data size {Features_norm.size()}")


def lasso_coordinate_descent(X, Y, lam, W, max_iter=1000, tol=1e-4):
    _, a = X.shape
    _, b = Y.shape
    # W shape = (a, b)

    summation_ = torch.sum(X ** 2, dim=0) # shape = (a,)
    # Adjust tolerance
    residual = Y - torch.matmul(X, W) # shape = (n, b)
    for iter in range(max_iter):
        W_old = W.clone()
        
        for k in range(a):
            
            if summation_[k] == 0:
                continue  
            
            # residual = Y - torch.matmul(X, W) # shape = (n, b)
            if torch.any(W[k,:] != 0):
                residual += torch.matmul(X[:, k].unsqueeze(-1), W[k, :].unsqueeze(0)) # shape = (n, b)
            
            
            # rho = torch.sum(X[:, k].unsqueeze(-1) * residual, dim=0) # shape = (b,)
            # rho = torch.mv(residual.T, X[:, k]) # shape = (b,)
            
            # cond1 = rho < -lam   
            # W[k, :][cond1] = (rho + lam)[cond1] # shape = (b,)
            
            # cond2 = rho > lam
            # W[k, :][cond2] = (rho - lam)[cond2]
            
            # cond3 = torch.abs(rho) <= lam
            # W[k, :][cond3] = 0
            
            rho = torch.mv(residual.T, X[:, k]).unsqueeze(-1) # shape = (b,1)
            W[k,:] = (torch.sign(rho) * torch.max( torch.abs(rho) - lam, torch.zeros_like(rho)) ).squeeze()
            
            
            W[k,:] /= summation_[k]
            
            if torch.any(W[k,:] != 0):
                residual -= torch.matmul(X[:, k].unsqueeze(-1), W[k, :].unsqueeze(0)) # shape = (n, b)
            
        # stopping criteria
        if ( torch.max(torch.abs(W - W_old)) < tol * torch.max(torch.abs(W)) ) and iter > 10:
            break
            
    print(f"Converged after {iter} iterations")
    return W


alpha = 0.078
alpha *= Embeddings_norm.size(0)
print(f"Alpha: {alpha}")

W = torch.zeros(Edim, Fdim).to(torch.float32).to(Embeddings_norm.device).T
dummymse_error = 0
for iter in range(1000):
    # Rotation
    Rotation = SVD(Features_norm, W, Embeddings_norm)
    
    # Lasso regression
    Y = torch.matmul(Embeddings_norm,Rotation)
    # W = torch.zeros(Edim, Fdim).to(torch.float32).to(Embeddings_norm.device).T
    W = lasso_coordinate_descent(Features_norm, Y, alpha, W)

    print(W.max(), W.min())
    mse, reg = MSE(Embeddings_norm, Features_norm, Rotation, W,  alpha / Embeddings_norm.size(0))
    # print(L1(Embeddings_norm, Features_norm, Rotation, W,  alpha))
    mse_error = mse + reg
    print(f"Iteration {iter} : MSE error {mse_error} MSE {mse} Reg {reg}")
    print(W)
    
    if abs(mse_error - dummymse_error) < 1e-6: 
        break
    else: 
        dummymse_error = mse_error




W = W.cpu().numpy()
np.savetxt(f"{output}/W.csv", W, delimiter=",")